## 1. Setup and GPU Check

**⚠️ IMPORTANT:** Enable GPU in Colab:  
Runtime → Change runtime type → Hardware accelerator → **T4 GPU**

In [ ]:
# Check GPU availability
!nvidia-smi

## 2. Install Dependencies

In [ ]:
!pip install -q ultralytics opencv-python-headless pyyaml scikit-learn tqdm

# Verify installation
import torch
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

from ultralytics import YOLO
print(f"✓ Ultralytics imported successfully")

## 3. Download Dataset

Download the Concrete Crack Images dataset from Kaggle.

In [ ]:
# Option 1: Download from Kaggle (requires Kaggle API setup)
# Uncomment if you have Kaggle API credentials:
# !pip install -q kagglehub
# import kagglehub
# from pathlib import Path
# import shutil
# dataset_cache = Path(kagglehub.dataset_download("jakubniemiec/concrete-crack-images"))
# DATASET_DIR = Path("/content/concrete-crack-images")
# if DATASET_DIR.exists():
#     shutil.rmtree(DATASET_DIR)
# shutil.copytree(dataset_cache, DATASET_DIR)

# Option 2: Manual upload
# If you don't have Kaggle API:
# 1. Download dataset from: https://www.kaggle.com/datasets/jakubniemiec/concrete-crack-images
# 2. Upload zip to Colab
# 3. Uncomment and run:
# !unzip -q concrete-crack-images.zip -d /content/concrete-crack-images

# For this example, using Kaggle API:
!pip install -q kagglehub
import kagglehub
from pathlib import Path
import shutil

print("Downloading dataset from Kaggle...")
dataset_cache = Path(kagglehub.dataset_download("jakubniemiec/concrete-crack-images"))
DATASET_DIR = Path("/content/concrete-crack-images")
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)
shutil.copytree(dataset_cache, DATASET_DIR)
print(f"✓ Dataset downloaded to: {DATASET_DIR}")

## 4. Data Preparation Code

Convert binary masks to YOLO polygon format.

In [ ]:
import cv2
import numpy as np
import yaml
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from pathlib import Path
from typing import List, Tuple

def mask_to_yolo_polygons(
    mask: np.ndarray,
    simplify_epsilon: float = 0.001,
    min_area: int = 50,
) -> List[np.ndarray]:
    """Convert binary mask to YOLO polygon format."""
    if mask.max() <= 1:
        mask = (mask * 255).astype(np.uint8)
    else:
        mask = mask.astype(np.uint8)
    
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    height, width = mask.shape
    polygons = []
    
    for contour in contours:
        area = cv2.contourArea(contour)
        if area < min_area:
            continue
        
        epsilon = simplify_epsilon * cv2.arcLength(contour, True)
        approx = cv2.approxPolyDP(contour, epsilon, True)
        
        if len(approx) < 3:
            continue
        
        polygon = approx.reshape(-1, 2).astype(np.float32)
        polygon[:, 0] /= width
        polygon[:, 1] /= height
        polygon = np.clip(polygon, 0, 1)
        polygons.append(polygon)
    
    return polygons

def create_yolo_label_file(polygons: List[np.ndarray], output_path: Path, class_id: int = 0):
    """Write YOLO format label file."""
    with open(output_path, "w") as f:
        for polygon in polygons:
            coords = " ".join(f"{x:.6f} {y:.6f}" for x, y in polygon)
            f.write(f"{class_id} {coords}\n")

def prepare_yolo_dataset(
    images_dir: Path,
    masks_dir: Path,
    output_dir: Path,
    train_ratio: float = 0.8,
    val_ratio: float = 0.1,
    test_ratio: float = 0.1,
    seed: int = 42,
):
    """Prepare complete YOLO dataset."""
    print("Preparing YOLO dataset...")
    
    # Get paired files
    image_extensions = {".jpg", ".jpeg", ".png", ".bmp"}
    images = {p.stem: p for p in images_dir.iterdir() if p.suffix.lower() in image_extensions}
    
    paired = []
    for stem, image_path in sorted(images.items()):
        mask_path = masks_dir / f"{stem}.png"
        if not mask_path.exists():
            mask_path = masks_dir / f"{stem}.jpg"
        if mask_path.exists():
            paired.append((image_path, mask_path))
    
    print(f"Found {len(paired)} image-mask pairs")
    
    # Split data
    indices = list(range(len(paired)))
    train_idx, temp_idx = train_test_split(indices, train_size=train_ratio, random_state=seed)
    val_size = val_ratio / (val_ratio + test_ratio)
    val_idx, test_idx = train_test_split(temp_idx, train_size=val_size, random_state=seed)
    
    splits = {
        "train": [paired[i] for i in train_idx],
        "val": [paired[i] for i in val_idx],
        "test": [paired[i] for i in test_idx],
    }
    
    print(f"Split: train={len(splits['train'])}, val={len(splits['val'])}, test={len(splits['test'])}")
    
    # Process each split
    for split_name, split_pairs in splits.items():
        images_out = output_dir / "images" / split_name
        labels_out = output_dir / "labels" / split_name
        images_out.mkdir(parents=True, exist_ok=True)
        labels_out.mkdir(parents=True, exist_ok=True)
        
        for image_path, mask_path in tqdm(split_pairs, desc=f"Processing {split_name}"):
            # Copy image
            shutil.copy2(image_path, images_out / image_path.name)
            
            # Convert mask to polygons
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            if mask is None:
                continue
            
            polygons = mask_to_yolo_polygons(mask)
            label_path = labels_out / f"{image_path.stem}.txt"
            create_yolo_label_file(polygons, label_path)
    
    # Create YAML file
    yaml_content = {
        "path": str(output_dir.resolve()),
        "train": "images/train",
        "val": "images/val",
        "test": "images/test",
        "names": {0: "crack"},
        "nc": 1,
    }
    
    yaml_path = output_dir / "crack-seg.yaml"
    with open(yaml_path, "w") as f:
        yaml.dump(yaml_content, f, default_flow_style=False, sort_keys=False)
    
    print(f"✓ Dataset prepared at: {output_dir}")
    print(f"✓ YAML config: {yaml_path}")
    return yaml_path

print("✓ Data preparation functions loaded")

## 5. Prepare Dataset

In [ ]:
IMAGES_DIR = DATASET_DIR / "images"
MASKS_DIR = DATASET_DIR / "masks"
YOLO_DIR = Path("/content/yolo_dataset")

yaml_path = prepare_yolo_dataset(
    images_dir=IMAGES_DIR,
    masks_dir=MASKS_DIR,
    output_dir=YOLO_DIR,
    train_ratio=0.8,
    val_ratio=0.1,
    test_ratio=0.1,
    seed=42,
)

## 6. Train YOLO Model

**Configuration:**
- Model: YOLOv8l-seg (large model)
- Resolution: 512px
- Epochs: 150 (change to 10 for quick test)
- Batch: 8 (adjust based on GPU memory)
- Optimizer: AdamW
- Optimized augmentation for crack preservation

In [ ]:
from ultralytics import YOLO
import torch

# Verify GPU is available
print(f"PyTorch CUDA available: {torch.cuda.is_available()}")
print(f"PyTorch CUDA device count: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"Current CUDA device: {torch.cuda.current_device()}")
    print(f"CUDA device name: {torch.cuda.get_device_name(0)}")

# Load model
model = YOLO("yolov8l-seg.pt")  # Downloads automatically

# Training configuration
results = model.train(
    data=str(yaml_path),
    epochs=150,  # Change to 10 for quick test
    batch=8,  # Reduce to 4 if GPU memory issues
    imgsz=512,
    patience=25,
    workers=2,
    device="cuda:0" if torch.cuda.is_available() else "cpu",  # Explicit GPU device
    seed=42,
    
    # Optimizer
    optimizer="AdamW",
    lr0=0.0001,
    lrf=0.1,
    weight_decay=0.0005,
    cos_lr=True,
    
    # Augmentation (optimized for crack preservation)
    mosaic=0.5,
    scale=0.2,
    translate=0.05,
    hsv_s=0.3,
    multi_scale=0.5,
    
    # Segmentation specific
    mask_ratio=1,  # Full resolution masks
    overlap_mask=True,
    close_mosaic=10,
    
    # Logging
    project="/content/runs/segment",
    name="cracktrack_yolov8l_512px",
    exist_ok=True,
    plots=True,
    save=True,
    verbose=True,
)

print("\n" + "="*70)
print("✓ Training complete!")
print("="*70)
print(f"Results saved to: {results.save_dir}")

## 7. Evaluation Code

In [ ]:
import json
import matplotlib.pyplot as plt

def load_ground_truth_mask(label_path: Path, image_shape: Tuple[int, int]) -> np.ndarray:
    """Load YOLO polygon labels and convert to binary mask."""
    height, width = image_shape
    mask = np.zeros((height, width), dtype=np.uint8)
    
    if not label_path.exists():
        return mask
    
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 7:
                continue
            coords = list(map(float, parts[1:]))
            points = np.array(coords).reshape(-1, 2)
            points[:, 0] *= width
            points[:, 1] *= height
            points = points.astype(np.int32)
            cv2.fillPoly(mask, [points], 255)
    
    return mask

def union_instance_masks(result) -> np.ndarray:
    """Union all instance masks from YOLO result."""
    if result.masks is None:
        return np.zeros(result.orig_shape[:2], dtype=np.uint8)
    
    masks = result.masks.data.cpu().numpy()
    if len(masks) == 0:
        return np.zeros(result.orig_shape[:2], dtype=np.uint8)
    
    orig_h, orig_w = result.orig_shape[:2]
    binary_mask = np.zeros((orig_h, orig_w), dtype=np.uint8)
    
    for mask in masks:
        resized_mask = cv2.resize(mask, (orig_w, orig_h), interpolation=cv2.INTER_LINEAR)
        binary_mask = np.maximum(binary_mask, (resized_mask > 0.5).astype(np.uint8) * 255)
    
    return binary_mask

def compute_dice_iou(pred_mask: np.ndarray, gt_mask: np.ndarray) -> Tuple[float, float]:
    """Compute Dice coefficient and IoU."""
    pred_binary = (pred_mask > 127).astype(np.uint8)
    gt_binary = (gt_mask > 127).astype(np.uint8)
    
    intersection = np.logical_and(pred_binary, gt_binary).sum()
    pred_sum = pred_binary.sum()
    gt_sum = gt_binary.sum()
    union = pred_sum + gt_sum - intersection
    
    if pred_sum + gt_sum == 0:
        dice = 1.0 if intersection == 0 else 0.0
    else:
        dice = 2.0 * intersection / (pred_sum + gt_sum)
    
    if union == 0:
        iou = 1.0 if intersection == 0 else 0.0
    else:
        iou = intersection / union
    
    return float(dice), float(iou)

def evaluate_model(model, data_root: Path, split: str = "test"):
    """Evaluate model on test set."""
    images_dir = data_root / "images" / split
    labels_dir = data_root / "labels" / split
    
    image_paths = sorted(images_dir.glob("*.png")) + sorted(images_dir.glob("*.jpg"))
    
    dice_scores = []
    iou_scores = []
    
    for image_path in tqdm(image_paths, desc=f"Evaluating {split}"):
        image = cv2.imread(str(image_path))
        if image is None:
            continue
        
        results = model.predict(source=str(image_path), imgsz=512, conf=0.25, verbose=False)
        pred_mask = union_instance_masks(results[0])
        
        label_path = labels_dir / f"{image_path.stem}.txt"
        gt_mask = load_ground_truth_mask(label_path, image.shape[:2])
        
        dice, iou = compute_dice_iou(pred_mask, gt_mask)
        dice_scores.append(dice)
        iou_scores.append(iou)
    
    metrics = {
        "dice_mean": float(np.mean(dice_scores)),
        "dice_std": float(np.std(dice_scores)),
        "iou_mean": float(np.mean(iou_scores)),
        "iou_std": float(np.std(iou_scores)),
        "num_images": len(dice_scores),
    }
    
    return metrics, dice_scores, iou_scores

print("✓ Evaluation functions loaded")

## 8. Run Evaluation

In [ ]:
# Load best model
best_model_path = Path(results.save_dir) / "weights" / "best.pt"
model_eval = YOLO(str(best_model_path))

# Evaluate on test set
metrics, dice_scores, iou_scores = evaluate_model(model_eval, YOLO_DIR, split="test")

print("\n" + "="*70)
print("EVALUATION RESULTS")
print("="*70)
print(f"Number of images: {metrics['num_images']}")
print(f"\nDice Coefficient: {metrics['dice_mean']:.4f} ± {metrics['dice_std']:.4f}")
print(f"IoU: {metrics['iou_mean']:.4f} ± {metrics['iou_std']:.4f}")

# Save metrics
metrics_file = Path(results.save_dir) / "test_metrics.json"
with open(metrics_file, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"\n✓ Metrics saved to: {metrics_file}")

## 9. Visualize Results

In [ ]:
# Plot metric distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(dice_scores, bins=30, edgecolor="black", alpha=0.7)
axes[0].axvline(metrics['dice_mean'], color="red", linestyle="--", linewidth=2, label=f"Mean={metrics['dice_mean']:.4f}")
axes[0].set_xlabel("Dice Coefficient")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Dice Distribution")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].hist(iou_scores, bins=30, edgecolor="black", alpha=0.7)
axes[1].axvline(metrics['iou_mean'], color="red", linestyle="--", linewidth=2, label=f"Mean={metrics['iou_mean']:.4f}")
axes[1].set_xlabel("IoU")
axes[1].set_ylabel("Frequency")
axes[1].set_title("IoU Distribution")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(Path(results.save_dir) / "metrics_distribution.png", dpi=150)
plt.show()

## 10. Download Results

Download the trained model and results to your local machine.

In [ ]:
# Zip results for download
!zip -r -q /content/results.zip {results.save_dir}

print(f"✓ Results zipped to: /content/results.zip")
print(f"\nDownload this file to get:")
print(f"  - Trained model weights (best.pt, last.pt)")
print(f"  - Training curves and plots")
print(f"  - Evaluation metrics")
print(f"\nIn Colab: Files → results.zip → Download")

# Show key files
print(f"\nKey model checkpoint: {best_model_path}")

## Summary

**Training Configuration:**
- Model: YOLOv8l-seg
- Resolution: 512px
- Epochs: 150 (or configured value)
- Augmentation: Optimized for crack preservation

**Expected Results:**
- Baseline IoU: 0.67
- Target IoU: 0.75-0.78
- Training time: ~8-10 hours on T4 GPU

**Next Steps:**
1. Download results.zip
2. Extract best.pt model
3. Use for inference or further training
4. Compare with local baseline results